## 작업형 1 핵심 유형  
작업형 1에서 반복해서 출제되는 조건 필터링, 정렬, 결측치 및 이상치 처리, 파생변수, 그룹별 집계, 날짜와 문자열 처리를 네 개의 데이터셋으로 연습해보자!

#### 1. 조건 필터링  
조건 필터링은 문제 조건에 맞는 행만 골라낸 뒤, 그 결과를 개수, 합계, 평균, 최대값 등을 계산하는 과정이다. 문제에서 ~인 데이터, ~이상인 행, ~에 해당하는 값처럼 조건이 제시되면 먼저 조건식을 만들고, 데이터프레임에 적용한다. 

In [6]:
import pandas as pd

filtered_df = pd.read_csv('./data/part02_type1/type1_filter_sort.csv')

print(filtered_df.shape)
print(filtered_df[['customer_id', 'age', 'region', 'score', 'purchase_count', 'total_spend', 'segment']].head().to_string(index=False))

(300, 8)
customer_id  age  region  score  purchase_count  total_spend segment
     FS0001   42 central   98.5               6       410000       A
     FS0002   39 central   98.5               4       390000       A
     FS0003   40   north   59.5              10       288900       B
     FS0004   42 central   45.4              10       154100       B
     FS0005   18   south   59.4               8       256400       B


In [8]:
# 조건에 맞는 행 선택
cond = filtered_df['region'] == 'central'
print(cond.head())
print(cond.sum())

0     True
1     True
2    False
3     True
4    False
Name: region, dtype: bool
78


In [9]:
cond = filtered_df['score'] >= 85
ans = filtered_df.loc[cond, 'total_spend'].sum()

print(cond.sum())
print(ans)

30
4990700


조건식 점검  

|조건 표현| 코드 |
|--|--|
|region이 east인 행|filter_df['region'] == 'east'|  
|age가 50 이상인 행|filter_df['age'] >= 50|  
|score가 60 미만인 행|filter_df['score'] < 60|

#### 2. 다중 조건 필터링  
1. 다중 조건 필터링이란  
다중 조건 필터링에선느 여러 조건을 연결해 원하는 데이터를 보다 구체적으로 선택 가능!  
조건을 연결할 때 & 은 당연히 and 연산자 |는 당연히 or 연산자!

In [16]:
cond = ((filtered_df['region'] == 'central') & (filtered_df['score'] >= 80))
ans = cond.sum()
print(ans)

17


In [17]:
cond = ((filtered_df['region'] == 'north') | (filtered_df['region'] == 'south'))
ans = cond.sum()
print(ans)

132


In [21]:
cond = (
    (filtered_df['segment'].isin(['A', 'B'])) & (filtered_df['total_spend'] >= 300000
))


ans = filtered_df.loc[cond, 'purchase_count'].sum()
print(cond.sum())
print(ans)

9
63


#### 정렬 후 값 추출  
1. 기본 정렬  
정렬 후 추출 값은 데이터를 특정 기준으로 줄 세운 뒤 원하는 위치의 값을 꺼내는 유형이다. 정렬 기준 칼럼, 오름차순 및 내림차순 방향,  
몇 번째 행, 마지막으로 출력할 컬럼을 차례대로 확인함

|기준|차순 | 사용코드 |  
|--|--|--|  
|가장 큰 값, 상위, 높은 값|내림차순|ascending=False|  
|가장 작은 값, 하위, 낮은 값|오름차순|ascending=True|

In [23]:
sorted_df = filtered_df.sort_values('total_spend', ascending=False)  
print(sorted_df[['customer_id', 'region', 'total_spend']].head())

    customer_id   region  total_spend
56       FS0057    south       501400
225      FS0226    south       436600
190      FS0191    north       415400
0        FS0001  central       410000
270      FS0271     west       405700


In [24]:
sorted_df = filtered_df.sort_values(
    ['score', 'total_spend'],
    ascending=[False, False]

)

print(sorted_df[['customer_id', 'score', 'total_spend']].head())

    customer_id  score  total_spend
225      FS0226  100.0       436600
104      FS0105  100.0       205300
257      FS0258  100.0       137500
0        FS0001   98.5       410000
1        FS0002   98.5       390000


In [ ]:
# 조건 적용 후 정렬 
central = filtered_df[filtered_df['region'] == 'central'] # region 칼럼에서 central인 애들만 뽑기
central = central.sort_values('total_spend', ascending=False) # central인 애들을 total_spend에 따라 내림차순 정렬

print(central[['customer_id', 'score', 'total_spend']].head())

    customer_id  score  total_spend
0        FS0001   98.5       410000
1        FS0002   98.5       390000
58       FS0059   66.4       377400
80       FS0081   54.9       334100
216      FS0217   80.1       317900


#### 인덱싱 기반 값 추출  
1. 인덱싱 기반 값 추출이란  
인덱싱 기반 값 추출은 특정 위치의 행이나 특정 인덱스의 값을 꺼내는 유형이다. 작업형 1에시는 조건을 적용하거나 정렬한 뒤  
'몇 번째 행의 특정 컬럼 값'을 요구하는 경우가 많다!  

|코드| 선택 결과|  
|--|--|  
|filter_df.iloc[0]| 현재 결과에서 첫 번째 위치의 행|  
|filter_df.iloc[4]| 현재 결과에서 다섯 번째 위치의 행|  
|filter_df.loc[4]| 인덱스 이름이 4인 행|  
|filter_df.iloc[2]['score']|세 번째 위치 행의 score 값|  
|filter_df.loc[4, 'score']|인덱스 4행의 score 값|

In [27]:
# 2. 원본 데이터 위치 선택
row = filtered_df.iloc[4] # 다섯 번째 행의 값
print(row[['customer_id', 'age', 'region', 'score', 'total_spend']])

customer_id    FS0005
age                18
region          south
score            59.4
total_spend    256400
Name: 4, dtype: object


필터링, 정렬 결과에서 위치 선택  
1. 필터링 결과에서 위치 선택  
필터링 결과에서 위치를 선택할 때는 iloc이 유용하다. score가 90 이상인 행을 골라낸 뒤 세 번째 위치 행의 customer_id로 확인해본다.

In [ ]:
high = filtered_df[filtered_df['score'] >= 90]

print(high[['customer_id', 'score', 'region']].head())
print(high.iloc[2]['customer_id']) # 여기서 2는 세번째 왜냐? 0부터 시작이니까

    customer_id  score   region
0        FS0001   98.5  central
1        FS0002   98.5  central
46       FS0047   91.9     east
93       FS0094   97.2  central
104      FS0105  100.0     east
FS0047


2. 정렬 결과에서 위치 선택  
정렬한 결과에서도 원래 인덱스는 유지된다. 첫 번째 iloc[0], 두 번째는 iloc[1], 세 번째는 iloc[2]로 선택한다

3. 인덱스 다시 붙이기  
필터링과 정렬을 거친 결과의 원본 번호가 뒤섞여 있다. 이때 reset_index(drop=True)를 사용하면 인덱스를 0부터 다시 붙일 수 있다.

In [29]:
north = filtered_df[filtered_df['region'] == 'north']
north = north.sort_values('score', ascending=False)
north = north.reset_index(drop=True)

print(north[['customer_id', 'score']].head())

  customer_id  score
0      FS0280   95.2
1      FS0043   88.4
2      FS0075   87.4
3      FS0206   86.3
4      FS0052   83.9


#### 결측치 처리  
결측치 처리 문제는 결측치를 확인하는 데서 끝나지 않고, 제거하거나 대체한 뒤 계산 결과를 요구한다.  
같은 데이터라도 처리 방식과 순서에 따라 답이 달라질 수 있으므로 문제의 지시를 그대로 따라야 한다.

#### 결측치 처리 과정

In [33]:
missing_df = pd.read_csv('./data/part02_type1/type1_missing_outlier.csv')

print(missing_df.shape)
print(missing_df.info())
print(missing_df[['row_id', 'item_group', 'price', 'qty', 'rating', 'views', 'event_flag']].head(10).to_string(index=False))

(350, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   row_id        350 non-null    object 
 1   item_group    350 non-null    object 
 2   price         350 non-null    float64
 3   qty           350 non-null    int64  
 4   rating        326 non-null    float64
 5   views         332 non-null    float64
 6   return_count  350 non-null    int64  
 7   event_flag    350 non-null    int64  
dtypes: float64(3), int64(3), object(2)
memory usage: 22.0+ KB
None
row_id item_group    price  qty  rating    views  event_flag
MO0001      alpha  58700.0   15     4.1  15225.0           1
MO0002      alpha  20300.0   11     NaN  20956.0           1
MO0003      delta  47600.0   10     3.0  14519.0           0
MO0004       beta 180000.0   14     3.8   7093.0           0
MO0005      delta  45900.0   15     NaN  22851.0           0
MO0006      alpha  4570

In [34]:
# 결측치 개수 확인
print(missing_df.isna().sum())
print(missing_df.isna().sum().sum())

row_id           0
item_group       0
price            0
qty              0
rating          24
views           18
return_count     0
event_flag       0
dtype: int64
42


In [41]:
# 결측치 대체  
filled = missing_df.copy()

median_rating = filled['rating'].median()
filled['rating'] = filled['rating'].fillna(median_rating)

print(median_rating)
print(round(filled['rating'].mean(), 2))

3.9
3.84


In [42]:
# 결측치 제거  
# 결측치가 있는 행을 제거할 때는 dropna()를 사용함
complete = missing_df.dropna(subset=['views'])

print(len(complete))
print(complete['qty'].sum())

332
3530


In [44]:
# 결측치 처리 순서  
filled = missing_df.copy()
filled['views'] = filled['views'].fillna(filled['views'].median())

cond = filled['qty'] >= 10
answer = int(filled.loc[cond, 'views'].mean())

print(answer)

15323


#### 그룹 기준 결측치 대체하기  
1. 그룹별 대푯값으로 결측치 대체하기  
전체 평균이나 중앙값으로 결측치를 채우면 모든 결측 행에 같은 값이 들어간다. 그룹마다 값의 분포가 다르다면 각 행이 속한 그룹의 대푯값을 사용해 결측치를 채울 수 있다.

In [48]:
group_median = missing_df.groupby('item_group')['rating'].transform('median') # transform의 결과는 원본과 같은 길이의 인덱스로 반환

filled = missing_df.copy()
filled['rating'] = filled['rating'].fillna(group_median) # 각 결측행에 해당 그룹의 중앙값이 들어가게 됨

print(filled['rating'].isna().sum())
print(filled.loc[missing_df['rating'].isna(), ['row_id', 'item_group', 'rating']].head(3).to_string(index=False))

0
row_id item_group  rating
MO0002      alpha     3.8
MO0005      delta     3.9
MO0009      gamma     3.9


2. 두 개 이상 기준으로 결측치 대체하기  
기준이 하나로 부족하면 여러 컬럼을 함께 묶을 수 있다. groupby()에 리스트로 컬럼을 넘기면 모든 기준이 같은 행끼리 하나의 그룹이 된다.  
기준이 세분화될수록 각 행에 더 가까운 값이 채워진다.

In [56]:
group_median = missing_df.groupby(['item_group', 'event_flag'])['rating'].transform('median') # 이벤트 플래그랑 아이템 그룹이 동일한 애들끼리 묶어서 그 그룹 내 median 값 가져오기!
#print(group_median)
filled = missing_df.copy()
filled['rating'] = filled['rating'].fillna(group_median)

print(filled['rating'].isna().sum())
print(filled.loc[missing_df['rating'].isna(), ['row_id', 'item_group', 'event_flag', 'rating']].head(3).to_string(index=False))

0
row_id item_group  event_flag  rating
MO0002      alpha           1    3.65
MO0005      delta           0    3.95
MO0009      gamma           1    4.25


#### 이상치 처리  
이상치 처리 후 계산은 이상치 기준을 구하고, 기준을 벗어난 값을 제외하거나 상한값으로 바꾼 뒤 평균, 합계 등을 계산하는 유형이다.  
처리 방식이 다르면 최종 답도 달라지므로 '제외'와 '대체'를 구분해야 한다

IQR 기준  
제1사분위수 Q1과 제3사분위수 Q3의 차이를 IQR이라고 한다. 일반적인 상한 기준은 Q3 + 1.5 * IQR, 하한 기준은 Q1 - 1.5 * IQR이다.  
이 예제에서는 price의 상한을 초과하는 큰 값만 제외하거나 상한값으로 대체한다.  

In [ ]:
q1 = missing_df['price'].quantile(0.25)
q3 = missing_df['price'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

normal = missing_df[missing_df['price'] <= upper]

print(q1, q3, iqr)
print(missing_df.shape[0], normal.shape[0]) # missing_df와 normal의 개수 비교, 3개가 이상값으로 날라감!
print(int(normal['price'].mean()))

26825.0 47500.0 20675.0
350 347
36778


#### 이상치 대체  
1. 상한값으로 이상치 대체  
이상치를 제외하는 대신에 상한값으로 눌러서 대체하는 방법도 있음. 상한을 초과하는 price를 upper로 바꿔본다.

In [58]:
capped = missing_df.copy()
capped.loc[capped['price'] > upper, 'price'] = upper

print(capped['price'].max())
print(round(capped['price'].mean(), 2))

78512.5
37136.39


2. 조건을 함께 적용하기  
이상치 처리 결과는 조건 계산으로 이어지는 경우가 많음. 상한 이하인 데이터에서 event_flag가 1인 행의 price 평균을 정수로 구해보자

In [60]:
normal = missing_df[missing_df['price'] <= upper]
cond = normal['event_flag'] == 1
ans = int(normal.loc[cond, 'price'].mean())
print(ans)

36631


#### 7. 파생변수 생성 후 계산  
1. 파생변수 생성  
파생변수는 기존 컬럼을 계산하거나 조건을 적용해 새로 만든 컬럼이다. 작업형 1에서는 '수량과 단가를 곱한 금액',  
'기준 이상이면 1, 아니면 0'처럼 새 값을 만든 뒤 합계, 평균, 개수를 묻는 문제가 자주 나온다.

이때 문풀 시 중간 계산 결과를 칼럼으로 남기는게 좋음  
복잡한 계산을 한 줄에 넣기 보단 gross_sales, discount_amount처럼 중간 결과를 새 컬럼으로 만들면 계산 흐름을 확인하기 쉽다.

In [62]:
sales_df = pd.read_csv('./data/part02_type1/type1_derived_group.csv')

print(sales_df.shape)
print(sales_df.info())
print(sales_df[['order_id', 'region', 'category', 'qty', 'unit_price', 'coupon_rate', 'net_sales']].head().to_string(index=False))

(500, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     500 non-null    object 
 1   store_id     500 non-null    object 
 2   region       500 non-null    object 
 3   category     500 non-null    object 
 4   qty          500 non-null    int64  
 5   unit_price   500 non-null    int64  
 6   coupon_rate  500 non-null    float64
 7   order_date   500 non-null    object 
 8   net_sales    500 non-null    int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 35.3+ KB
None
order_id   region   category  qty  unit_price  coupon_rate  net_sales
 DG00001 mountain       food    9       37000         0.00     333000
 DG00002    metro    digital    9       60900         0.15     465885
 DG00003    coast stationery   11       26300         0.00     289300
 DG00004    river    digital    4       29400         0.00     117600
 DG0000

In [63]:
# 계산형 파생변수
sales_df['gross_sales'] = sales_df['qty'] * sales_df['unit_price'] # 판매 개수 * 1개 당 가격 = 판매 총액
print(sales_df[['order_id', 'qty', 'unit_price', 'gross_sales']].head())


  order_id  qty  unit_price  gross_sales
0  DG00001    9       37000       333000
1  DG00002    9       60900       548100
2  DG00003   11       26300       289300
3  DG00004    4       29400       117600
4  DG00005    7       41500       290500


In [65]:
# 정수형 파생변수 
import numpy as np

sales_df['gross_sales_10k'] = (np.floor(sales_df['gross_sales'] / 10000).astype(int))

print(sales_df[['order_id', 'gross_sales', 'gross_sales_10k']].head().to_string(index=False))

order_id  gross_sales  gross_sales_10k
 DG00001       333000               33
 DG00002       548100               54
 DG00003       289300               28
 DG00004       117600               11
 DG00005       290500               29


In [ ]:
# 파생 변수 계산  
# 할인 금액 계산 
sales_df['discount_amount'] = (sales_df['gross_sales'] * sales_df['coupon_rate']) # 전체 판매량 * 쿠폰 할인율
sales_df['calc_net_sales'] = (sales_df['gross_sales'] - sales_df['discount_amount']) # 전체 판배량에서 할인된 금액 차감

print(int(sales_df['discount_amount'].sum()))

8912475


In [68]:
# 조건형 파생변수
sales_df['high_order'] = (sales_df['gross_sales'] >= 500000).astype(int)

print(sales_df['high_order'].sum())
print(sales_df.groupby('category')['high_order'].sum().sort_values(ascending=False))

119
category
stationery    30
food          24
hobby         23
digital       21
home          21
Name: high_order, dtype: int32


In [ ]:
# 파생변수를 이용한 계산
cond = sales_df['region'] == 'metro' # 지역이 메트로인 행 선택
answer = sales_df.loc[cond, 'gross_sales'].sum()  # 메트로인 행의 gross_sales 계산
print(answer)

61834000


#### 수치 범위를 구간으로 나누기  
수량이나 점수처럼 연속된 숫자를 일정한 범위로 나누면 범주형 파생변수를 만들 수 있다.  
pd.cut()의 bins에는 구간 경계를, labes에는 각 구간에 붙일 이름을 지정한다.

In [72]:
# 수치 범위를 구간으로 나누기
bins = [0, 4, 8, 12, np.inf]
labels = ['3개 이하', '4~7개', '8~11개', '12개 이상' ]

sales_df['qty_group'] = pd.cut(sales_df['qty'], bins=bins, labels=labels, right=False) # right가 False이므로 오른쪽 경계는 제외함
qty_summary = (sales_df.groupby('qty_group', observed=False)['net_sales'].agg(['count', 'mean']).round(1))

print(qty_summary)
print(sales_df['qty_group'].isna().sum())

           count      mean
qty_group                 
3개 이하        105   77957.0
4~7개         152  231105.6
8~11개        131  367340.5
12개 이상       112  594233.0
0


계획값과 실제값 차이 비교  
계획값과 실젯값의 차이는 실젯값이 더 크면 양수, 더 작으면 음수가 된다. 어느 방향으로 벗어났는지보다 차이의 크기가  
중요하다면 abs()로 절댓값을 구한 뒤 idxmax()를 사용한다.

In [75]:
gap_df = pd.DataFrame({
    'branch' : ['A지점', 'B지점', 'C지점', 'D지점'],
    'planned_orders' : [320, 280, 350, 300],
    'actual_orders' : [305, 310, 330, 260]
})

gap_df['gap'] = (gap_df['actual_orders'] - gap_df['planned_orders'])

idx = gap_df['gap'].abs().idxmax()
answer = gap_df.loc[idx, 'branch']
print(gap_df.to_string(index=False))
print(answer)

branch  planned_orders  actual_orders  gap
   A지점             320            305  -15
   B지점             280            310   30
   C지점             350            330  -20
   D지점             300            260  -40
D지점
